# Emergency Room — PySpark Cluster Test Notebook

This notebook verifies connectivity from **JupyterLab** to:
1. **Apache Spark Cluster** (`spark://spark-master:7077`)
2. **Datasets & Data Lake** (`/opt/dataset`, `/opt/data/bronze`, `/opt/data/silver`)
3. **PostgreSQL Gold Layer** (`jdbc:postgresql://postgres:5432/er_data_warehouse`)

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ER_Jupyter_Spark_Test")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark master :", spark.sparkContext.master)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/27 21:00:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.3
Spark master : spark://spark-master:7077


In [ ]:
# Test reading sample CSV from /opt/dataset
df = spark.read.csv("/opt/dataset/er_sample_data.csv", header=True, inferSchema=True)
print("Row count:", df.count())
df.show(5, truncate=False)

[Stage 0:>                                                          (0 + 0) / 1]

In [ ]:
# Test PostgreSQL JDBC connection (Gold Layer)
pg_df = (
    spark.read.format("jdbc")
    .option("url", "jdbc:postgresql://postgres:5432/er_data_warehouse")
    .option("dbtable", "information_schema.tables")
    .option("user", "er_admin")
    .option("password", "er_password")
    .option("driver", "org.postgresql.Driver")
    .load()
)
pg_df.filter(pg_df.table_schema == "public").select("table_name").show()